# 🧪 Лабораторна робота №4
## Тема: Шини I2C та SPI. Підключення периферійних пристроїв (дисплей, датчики)

## 🎯 Мета роботи:
- Зрозуміти принципи роботи шин I2C та SPI і чим вони відрізняються від UART
- Навчитися сканувати шину I2C та знаходити адреси підключених пристроїв
- Вивести інформацію на OLED-дисплей SSD1306 через I2C
- Підключити периферійний пристрій через SPI (матричний дисплей MAX7219)
- Побудувати міні-проєкт, що поєднує кілька інтерфейсів разом

---

## 📖 Розширені теоретичні відомості

### 1️⃣ Шина I2C
- **I2C (Inter-Integrated Circuit)** — послідовна шина всього на **двох дротах**: `SDA` (дані) і `SCL` (тактування), до якої можна підключити багато пристроїв одночасно.
- Кожен пристрій на шині має свою унікальну **адресу** (зазвичай 7-бітну, наприклад `0x3C` для типового OLED-дисплея).
- Обидва дроти потребують підтягувальних резисторів (pull-up) до `+3.3V` — на готових модулях (breakout-платах) вони зазвичай вже вбудовані.
- На ESP32 типові піни для I2C: `SDA = GPIO21`, `SCL = GPIO22` (можна призначити й інші).

### 2️⃣ Клас `machine.I2C`
```python
from machine import I2C, Pin

i2c = I2C(0, scl=Pin(22), sda=Pin(21), freq=400000)

devices = i2c.scan()          # список знайдених адрес (int)
print([hex(d) for d in devices])

i2c.writeto(0x3C, b'\x00\xAE')   # записати байти за адресою пристрою
data = i2c.readfrom(0x3C, 2)       # прочитати N байтів
```

- `scan()` — перше, що варто зробити при підключенні нового пристрою: він поверне список адрес усіх, хто відповів на шині.
- Для готових пристроїв зазвичай використовують готові бібліотеки-драйвери (наприклад `ssd1306.py` для OLED), а не пишуть протокол вручну.

### 3️⃣ OLED-дисплей SSD1306 через I2C
- Потрібна бібліотека `ssd1306.py` (входить до стандартних прикладів MicroPython; у Wokwi вона доступна автоматично при виборі відповідної деталі, або її можна додати як окремий файл проєкту).
```python
from machine import I2C, Pin
import ssd1306

i2c = I2C(0, scl=Pin(22), sda=Pin(21))
oled = ssd1306.SSD1306_I2C(128, 64, i2c)   # роздільність 128x64

oled.fill(0)                    # очистити екран (0 = чорний)
oled.text("Hello, ESP32!", 0, 0)
oled.show()                     # обов'язково — «намалювати» буфер на екрані
```

- Основні методи: `fill()`, `text(рядок, x, y)`, `pixel(x, y, колір)`, `line()`, `rect()`, `fill_rect()` — і завжди завершувати `show()`.

### 4️⃣ Шина SPI
- **SPI (Serial Peripheral Interface)** — швидша шина, що використовує 4 лінії: `SCK` (тактування), `MOSI` (Master Out Slave In — дані від плати до пристрою), `MISO` (Master In Slave Out — дані від пристрою до плати), і окремий `CS`/`SS` (Chip Select) **для кожного** підключеного пристрою.
- На відміну від I2C, де адреса передається в самих даних, у SPI пристрій обирається фізичним сигналом на його лінії `CS`.
- SPI зазвичай швидший за I2C — використовується для дисплеїв (TFT), SD-карток, деяких датчиків.

### 5️⃣ Клас `machine.SPI` — приклад із матричним дисплеєм MAX7219
```python
from machine import SPI, Pin
import max7219   # бібліотека-драйвер для MAX7219 (8x8 LED-матриця)

spi = SPI(1, baudrate=10000000, polarity=0, phase=0,
          sck=Pin(18), mosi=Pin(23), miso=Pin(19))
cs = Pin(5, Pin.OUT)

display = max7219.Matrix8x8(spi, cs, 1)   # 1 = кількість модулів матриці
display.brightness(2)
display.fill(0)
display.text('Hi', 0, 0, 1)
display.show()
```

- Кожен новий SPI-пристрій на тій самій шині отримує **свій** пін `CS`, а `SCK`/`MOSI`/`MISO` — спільні для всіх.

### 6️⃣ Порівняння інтерфейсів
| Інтерфейс | Дротів | Швидкість | Адресація | Типове застосування |
|---|---|---|---|---|
| UART | 2 (TX/RX) | низька-середня | немає (точка-точка) | GPS, зв'язок між платами |
| I2C | 2 (SDA/SCL) | середня | адреса в даних | дисплеї, датчики, RTC-годинники |
| SPI | 4+ (SCK/MOSI/MISO/CS×N) | висока | окремий CS на пристрій | дисплеї TFT, SD-картки, швидкі АЦП |

### 7️⃣ Типові помилки початківців
- `i2c.scan()` повертає порожній список → переплутані `SDA`/`SCL`, немає підтягувальних резисторів, або пристрій підключений без живлення.
- Забули викликати `oled.show()` після малювання → на екрані нічого не з'являється (дані лишились тільки в буфері).
- На SPI кілька пристроїв ділять `CS` замість того, щоб мати окремий пін кожен → конфлікти на шині.
- Переплутали `MOSI`/`MISO` місцями.
- Використовують адресу I2C-пристрою «на око» замість того, щоб спершу виконати `scan()`.

### 8️⃣ Як перевіряти код цієї лабораторної роботи

У MicroPython немає локального "пісочного" середовища, як для звичайного Python — код виконується **на самому мікроконтролері** ESP32. Є два способи перевірити свої рішення.

**Варіант A. Jupyter Notebook + реальна плата ESP32 (рекомендовано, якщо є плата)**

1. Прошийте плату прошивкою MicroPython ([https://micropython.org/download/ESP32_GENERIC/](https://micropython.org/download/ESP32_GENERIC/)), один раз через `esptool.py`.
2. Встановіть ядро для Jupyter, яке вміє працювати з платою по USB/Serial:
   ```bash
   pip install jupyter_micropython_kernel
   python -m jupyter_micropython_kernel.install
   ```

3. Відкрийте цей ноутбук і оберіть ядро **MicroPython** (або **MicroPython - USB**) замість Python 3.
4. У першій комірці виконайте підключення до плати (замініть порт на свій — `COMx` у Windows, `/dev/ttyUSB0` або `/dev/ttyACM0` у Linux/macOS):
   ```
   %serialconnect --port=COM5 --baud=115200
   ```

5. Далі виконуйте комірки з кодом як завжди — код одразу запускається на платі, а `print()` повертає результат прямо в ноутбук. Світлодіоди й кнопки реагують фізично.
6. Якщо потрібно скинути з'єднання: `%disconnect`; список файлів на платі показує `%ls`, а перелік усіх доступних магічних команд ядра — `%lsmagic`.

**Варіант B. Немає плати під рукою — симулятор Wokwi**

1. Відкрийте [https://wokwi.com](https://wokwi.com) → **New Project → ESP32 → MicroPython**.
2. У файлі `main.py` вставте код із завдання.
3. На вкладці **diagram.json** опишіть схему (перелік деталей і з'єднань наведено в кожному завданні нижче) — або зберіть її мишкою в графічному редакторі Wokwi, перетягуючи компоненти з панелі **Parts**.
4. Натисніть зелену кнопку **▶ Play** — почнеться симуляція. Вивід `print()` з'являється у вкладці **Serial Monitor**.
5. Перевірте результат за критеріями, вказаними у кожному завданні (що саме має блимати/показуватись/виводитись).
6. Готовий проєкт можна зберегти (**Save**) і надіслати викладачу посилання на нього — це і буде звіт з лабораторної.

**Схема для цієї лабораторної (Wokwi):** OLED SSD1306 (128×64, I2C) — `SDA→GPIO21`, `SCL→GPIO22`, живлення `3V3`/`GND`; потенціометр на `GPIO34` (як у Лабораторній №2); матричний дисплей MAX7219 (8x8 LED Matrix, SPI) — `DIN(MOSI)→GPIO23`, `CLK(SCK)→GPIO18`, `CS→GPIO5`, живлення `5V`/`GND`.

*Важливо для Wokwi:* компоненти OLED і MAX7219 у бібліотеці Wokwi вже «знають» стандартні MicroPython-драйвери (`ssd1306`, `max7219`) — достатньо додати `import ssd1306` / `import max7219` у `main.py`, встановлювати їх окремо не потрібно. При роботі з реальною платою поза Wokwi файли драйверів (`ssd1306.py`, `max7219.py`) потрібно попередньо скопіювати на плату через `mpremote cp` / Thonny.

> 💡 Порада: пишіть і перевіряйте код невеликими фрагментами — спочатку переконайтесь, що плата взагалі відповідає (наприклад, `print("hello")`), і лише потім додавайте логіку.

---

## 📝 Завдання 1. Сканування шини I2C
Ініціалізуйте I2C на пінах `SDA=21, SCL=22` і виведіть список знайдених адрес пристроїв у шістнадцятковому форматі.

In [ ]:
from machine import I2C, Pin
import time

i2c = I2C(0, scl=Pin(22), sda=Pin(21), freq=400000)

while True:
    devices = i2c.scan()
    if devices:
        print("Знайдено пристроїв:", [hex(d) for d in devices])
    else:
        print("Пристроїв не знайдено — перевірте підключення")
    time.sleep(2)


### 🔢 Завдання:
1. Виведіть кількість знайдених пристроїв окремим рядком.
2. Додайте світлодіод, що вмикається, якщо на шині знайдено хоча б один пристрій.
3. Порівняйте знайдену адресу з очікуваною адресою OLED (`0x3C`) і виведіть, чи це саме він.
4. Виведіть адреси не лише в hex, а й у десятковому форматі поруч.
5. Змініть `freq` на `100000` (стандартний режим замість швидкого) і перевірте, чи сканування досі працює.
6. Обгорніть сканування в `try/except` і виведіть зрозуміле повідомлення при помилці шини.
7. Реалізуйте повторне сканування лише по натисканню кнопки, а не в циклі кожні 2 секунди.
8. Додайте таймер (`machine.Timer`) для періодичного сканування замість `time.sleep()`.
9. Виведіть список адрес одним рядком через кому замість стандартного виводу списку Python.
10. Порахуйте й виведіть, скільки разів адреса `0x3C` з'являлась за 10 сканувань поспіль.

## 📝 Завдання 2. Текст на OLED-дисплеї
Виведіть на OLED SSD1306 (128×64) три рядки тексту: своє ім'я, назву роботи та номер варіанту/групи.

In [ ]:
from machine import I2C, Pin
import ssd1306

i2c = I2C(0, scl=Pin(22), sda=Pin(21))
oled = ssd1306.SSD1306_I2C(128, 64, i2c)

oled.fill(0)
oled.text("ESP32 + MicroPython", 0, 0)
oled.text("Lab work 4", 0, 16)
oled.text("Group: IO-21", 0, 32)
oled.show()


### 🔢 Завдання:
1. Додайте четвертий рядок з поточним значенням якогось лічильника, що оновлюється щосекунди.
2. Намалюйте прямокутну рамку (`rect`) навколо всього тексту.
3. Виведіть у центрі екрана велике число (наприклад, лічильник натискань кнопки) великим шрифтом (масштабуванням через кілька `text()` поруч).
4. Реалізуйте біжучий рядок (текст рухається зліва направо і зникає за межею екрана).
5. Намалюйте просту геометричну фігуру (`fill_rect`, `line`) поруч із текстом.
6. Виведіть прогрес-бар (`fill_rect` змінної довжини) на основі значення потенціометра (ADC).
7. Реалізуйте перемикання між двома «екранами» (сторінками) кнопкою.
8. Додайте годинник (лічильник секунд від старту програми) у правому верхньому куті.
9. Виведіть таблицю з двох стовпців (наприклад «Пін» / «Стан») для трьох різних входів.
10. Реалізуйте просте меню з 3 пунктів, де поточний обраний пункт підсвічується інвертованим кольором (`fill_rect` + `text` з кольором фону).

## 📝 Завдання 3. Проєкт: потенціометр + прогрес-бар на OLED
Зчитуйте потенціометр (ADC, `GPIO34`) і відображайте його значення на OLED у вигляді числа (%) та горизонтального прогрес-бару.

In [ ]:
from machine import I2C, Pin, ADC
import ssd1306
import time

i2c = I2C(0, scl=Pin(22), sda=Pin(21))
oled = ssd1306.SSD1306_I2C(128, 64, i2c)

pot = ADC(Pin(34))
pot.atten(ADC.ATTN_11DB)

while True:
    percent = int(pot.read_u16() / 65535 * 100)

    oled.fill(0)
    oled.text("Value: {}%".format(percent), 0, 0)
    oled.rect(0, 20, 100, 10, 1)                       # рамка прогрес-бару
    bar_width = int(percent / 100 * 98)
    oled.fill_rect(1, 21, bar_width, 8, 1)              # заповнення
    oled.show()

    time.sleep_ms(100)


### 🔢 Завдання:
1. Додайте вертикальний прогрес-бар поруч із горизонтальним.
2. Виведіть окремо мінімальне й максимальне значення, зафіксовані за весь час роботи.
3. Додайте кольорову (текстову) відмітку рівня: «Низький» / «Середній» / «Високий».
4. Об'єднайте з керуванням яскравістю LED (як у Лабораторній №2, Завдання 3) — і LED, і OLED реагують на той самий потенціометр.
5. Реалізуйте плавну (анімовану) зміну прогрес-бару замість миттєвого стрибка при швидкому русі потенціометра.
6. Додайте другий потенціометр і другий прогрес-бар нижче першого.
7. Виведіть графік (просту лінію history) з останніх 50 значень у нижній частині екрана.
8. Додайте звуковий сигнал зумером, коли значення перевищує 90%.
9. Реалізуйте кнопку «заморозити» показання на екрані (пауза оновлення).
10. Замініть числове значення (%) на «одиниці виміру» на вибір: імітуйте, що це температура (0-100 → 0-40 °C).

## 📝 Завдання 4. Матричний дисплей MAX7219 (SPI)
Підключіть 8×8 LED-матрицю MAX7219 через SPI (`SCK=18, MOSI=23, CS=5`) і виведіть на ній цифру, що збільшується від 0 до 9 кожні дві секунди.

In [ ]:
from machine import SPI, Pin
import max7219
import time

spi = SPI(1, baudrate=10000000, polarity=0, phase=0,
          sck=Pin(18), mosi=Pin(23), miso=Pin(19))
cs = Pin(5, Pin.OUT)

display = max7219.Matrix8x8(spi, cs, 1)
display.brightness(4)

for digit in range(10):
    display.fill(0)
    display.text(str(digit), 0, 0, 1)
    display.show()
    time.sleep(2)


### 🔢 Завдання:
1. Реалізуйте зворотний відлік від 9 до 0.
2. Виведіть буквений текст (наприклад, ваші ініціали), що прокручується зліва направо (біжучий рядок).
3. Керуйте яскравістю матриці (`brightness()`) потенціометром.
4. Реалізуйте просту анімацію (наприклад, «серце», що з'являється і зникає) через прямі виклики `pixel()`.
5. Додайте кнопку, яка запускає/зупиняє прокрутку тексту.
6. Виведіть на матриці поточний лічильник натискань кнопки (як у Лабораторній №1).
7. Реалізуйте індикатор рівня (стовпчикова діаграма на 8 стовпців) на основі значення ADC.
8. З'єднайте два модулі матриці послідовно (якщо доступно в симуляторі) і виведіть текст, що займає обидва.
9. Реалізуйте просту гру «змійка, що рухається по колу» (по периметру матриці).
10. Додайте синхронізацію: коли на OLED (Завдання 2) змінюється лічильник, те саме число одночасно з'являється на матриці.

## 📝 Завдання 5. Міні-проєкт: панель приладів (OLED + UART)
Об'єднайте вивчене: приймайте по UART (`UART2`, `TX=GPIO17`, `RX=GPIO16` — як у Лабораторній №3, Завдання 5) рядки формату `TEMP:23.4`, розбирайте їх і виводьте актуальне значення температури та історію останніх 3 значень на OLED-дисплеї.

In [ ]:
from machine import I2C, Pin, UART
import ssd1306
import time

i2c = I2C(0, scl=Pin(22), sda=Pin(21))
oled = ssd1306.SSD1306_I2C(128, 64, i2c)
uart = UART(2, baudrate=115200, tx=17, rx=16)   # окремий UART, не той, що зайнятий REPL

history = []

def parse_temp(line):
    try:
        text = line.decode().strip()
        if text.startswith("TEMP:"):
            return float(text.split(":")[1])
    except (ValueError, IndexError):
        pass
    return None

while True:
    if uart.any():
        line = uart.readline()
        value = parse_temp(line) if line else None
        if value is not None:
            history.append(value)
            history = history[-3:]        # тримаємо лише останні 3 значення

            oled.fill(0)
            oled.text("Temp: {:.1f} C".format(value), 0, 0)
            oled.text("History:", 0, 20)
            for i, v in enumerate(history):
                oled.text("{:.1f}".format(v), 0, 32 + i * 10)
            oled.show()

    time.sleep_ms(50)


### 🔢 Завдання:
1. Надішліть тестові дані з другої плати (код передавача з Лабораторної №3, Завдання 5, змінений так, щоб надсилати рядки `TEMP:...` замість `COUNT:...`) і перевірте прийом.
2. Додайте на екран мінімальне й максимальне значення з усієї історії (не лише останніх 3).
3. Виводьте попередження на екрані («Увага!»), якщо значення виходить за межі 15-35 градусів.
4. Додайте другий тип даних (наприклад `HUM:60.0` для вологості) з окремим полем на екрані.
5. Реалізуйте світлодіодну індикацію статусу зв'язку (LED блимає, поки дані надходять регулярно; гасне, якщо дані не приходять > 5 с).
6. Намалюйте простий графік (лінію) останніх 10-20 значень температури внизу екрана.
7. Додайте зумер, що подає короткий сигнал при кожному новому отриманому значенні.
8. Реалізуйте збереження історії значень у список довільної довжини (не обмежений 3) і виведення середнього.
9. Додайте на дисплей лічильник загальної кількості отриманих повідомлень від старту програми.
10. Опишіть у markdown-комірці, як цей проєкт можна розширити: логування на SD-картку через SPI, передача даних по Wi-Fi тощо.

---

## Висновок

> В цій ...